## Cell 0 — Install

In [1]:
!pip install -q "transformers==4.57.1" "peft==0.18.0" accelerate bitsandbytes "pandas==2.2.2" "pillow==11.3.0" tqdm kagglehub

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 130.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 556.4/556.4 kB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 20.5 MB/s eta 0:00:00


## Cell 1 — Imports/data/model

In [2]:
import os, json, random, gc
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
from transformers import AutoProcessor, AutoModelForVision2Seq
from peft import PeftModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

MODEL_ID = "HuggingFaceTB/SmolVLM-500M-Instruct"
CHOICE_LETTERS = "ABCDEFGHIJ"

In [3]:
import kagglehub
kagglehub.login()

Kaggle credentials set.
Kaggle credentials successfully validated.


In [4]:
COMP_DIR = Path(kagglehub.competition_download("pixels-to-predictions"))

DATA_DIR = COMP_DIR / "data"
if not DATA_DIR.exists():
    DATA_DIR = COMP_DIR

100%|██████████| 358M/358M [00:18<00:00, 20.3MB/s]

Extracting files...


In [5]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

example_name = Path(val_df.iloc[0]["image_path"]).name
DATA_ROOT = list(COMP_DIR.rglob(example_name))[0].parents[2]

from google.colab import drive
drive.mount("/content/drive")

ADAPTER_DIR = "/content/drive/MyDrive/lora_v9_r8_all7_flatlr_trainval"

processor = AutoProcessor.from_pretrained(ADAPTER_DIR)
processor.tokenizer.padding_side = "left"

base_model = AutoModelForVision2Seq.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="eager",
)

model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
model.eval()

print("Loaded v9")

Mounted at /content/drive


/usr/local/lib/python3.12/dist-packages/transformers/models/auto/modeling_auto.py:2284: FutureWarning: The class `AutoModelForVision2Seq` is deprecated and will be removed in v5.0. Please use `AutoModelForImageTextToText` instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.02G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

Loaded v9


## Cell 2 — Inference helpers

In [6]:
def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def img_384(row):
    path = DATA_ROOT / row["image_path"]
    if not path.exists():
        path = DATA_ROOT.parent / row["image_path"]

    img = Image.open(path).convert("RGB")
    img.thumbnail((384, 384), Image.BICUBIC)
    return img

def get_candidate_token_ids(tokenizer):
    candidate_ids = {}
    for letter in CHOICE_LETTERS:
        forms = [letter, " " + letter, "\n" + letter, letter + ".", " " + letter + "."]
        ids = set()
        for form in forms:
            enc = tokenizer.encode(form, add_special_tokens=False)
            if enc:
                ids.add(enc[-1])
        candidate_ids[letter] = sorted(ids)
    return candidate_ids

candidate_ids = get_candidate_token_ids(processor.tokenizer)

## Cell 3 — Original v9 prompt + metadata prompt

In [7]:
def build_prompt_original(row):
    choices = row["choices"]
    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {choice}"
        for i, choice in enumerate(choices)
    )

    question = clean_text(row.get("question", ""))
    hint = clean_text(row.get("hint", ""))
    lecture = clean_text(row.get("lecture", ""))

    if lecture:
        lecture = lecture[:400]
    if hint:
        hint = hint[:400]

    prompt = "<image>\n"
    prompt += "Answer the multiple-choice question.\n\n"

    if lecture:
        prompt += f"Lecture: {lecture}\n\n"
    if hint:
        prompt += f"Hint: {hint}\n\n"

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"
    return prompt


def build_prompt_metadata(row):
    prompt = "<image>\n"
    prompt += "Answer the multiple-choice question.\n\n"

    subject = clean_text(row.get("subject", ""))
    topic = clean_text(row.get("topic", ""))
    skill = clean_text(row.get("skill", ""))

    if subject:
        prompt += f"Subject: {subject}\n"
    if topic:
        prompt += f"Topic: {topic}\n"
    if skill:
        prompt += f"Skill: {skill}\n"

    prompt += "\n"

    lecture = clean_text(row.get("lecture", ""))
    hint = clean_text(row.get("hint", ""))
    question = clean_text(row.get("question", ""))

    if lecture:
        prompt += f"Lecture: {lecture[:400]}\n\n"
    if hint:
        prompt += f"Hint: {hint[:400]}\n\n"

    choices_text = "\n".join(
        f"{CHOICE_LETTERS[i]}. {choice}"
        for i, choice in enumerate(row["choices"])
    )

    prompt += f"Question: {question}\n\n"
    prompt += f"Choices:\n{choices_text}\n\n"
    prompt += "Answer:"
    return prompt

## Cell 4 — Prediction function

In [8]:
def predict_score_one(row, prompt_fn):
    image = img_384(row)
    prompt = prompt_fn(row)

    inputs = processor(
        text=[prompt],
        images=[image],
        return_tensors="pt",
    )

    inputs = {
        k: v.to(model.device) if torch.is_tensor(v) else v
        for k, v in inputs.items()
    }

    with torch.inference_mode():
        logits = model(**inputs).logits[:, -1, :]

    log_probs = torch.log_softmax(logits, dim=-1)

    scores = []
    for choice_idx in range(len(row["choices"])):
        letter = CHOICE_LETTERS[choice_idx]
        scores.append(log_probs[0, candidate_ids[letter]].max().item())

    return int(np.argmax(scores)), scores

## Cell 5 — Create original-v9 reproduction submission

In [9]:
preds_original = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test original"):
    pred, _ = predict_score_one(row, build_prompt_original)
    preds_original.append(pred)
    torch.cuda.empty_cache()

sub_original = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds_original,
})

sub_original.to_csv("/content/submission_v9_original_repro.csv", index=False)
sub_original.to_csv("/content/drive/MyDrive/submission_v9_original_repro.csv", index=False)

print(sub_original["answer"].value_counts().sort_index())

Test original:   0%|          | 0/1008 [00:00<?, ?it/s]

answer
0    364
1    332
2    206
3     90
4     16
Name: count, dtype: int64


## Cell 6 — Create metadata-prompt submission

In [10]:
preds_meta = []

for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Test metadata"):
    pred, _ = predict_score_one(row, build_prompt_metadata)
    preds_meta.append(pred)
    torch.cuda.empty_cache()

sub_meta = pd.DataFrame({
    "id": test_df["id"],
    "answer": preds_meta,
})

sub_meta.to_csv("/content/submission.csv", index=False)
sub_meta.to_csv("/content/drive/MyDrive/submission_v9_metadata_prompt.csv", index=False)

print(sub_meta["answer"].value_counts().sort_index())
print("Saved metadata submission")

Test metadata:   0%|          | 0/1008 [00:00<?, ?it/s]

answer
0    380
1    324
2    206
3     82
4     16
Name: count, dtype: int64
Saved metadata submission


## Cell 7 — Compare disagreement

In [11]:
diff = np.mean(np.array(preds_original) != np.array(preds_meta))
print("Disagreement rate:", diff)

compare_df = pd.DataFrame({
    "id": test_df["id"],
    "original": preds_original,
    "metadata": preds_meta,
})

display(compare_df[compare_df["original"] != compare_df["metadata"]].head(30))

Disagreement rate: 0.0248015873015873


,id,original,metadata
16,test_00347,3,4
18,test_02971,3,0
31,test_02616,1,0
34,test_03240,4,0
185,test_00397,2,0
265,test_02445,2,0
275,test_04069,1,2
303,test_01888,2,0
469,test_03520,2,1
486,test_02161,1,0


In [13]:
from google.colab import files
files.download("/content/submission.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>